# Fine-Tuning an LLM with PEFT (LoRA / QLoRA) on Google Colab

This notebook walks through **parameter-efficient fine-tuning (PEFT)** of a small open-source language model on a free Google Colab GPU (T4).

### What you'll do
1. Load a base model in 4-bit precision (QLoRA)
2. Attach trainable LoRA adapters instead of updating all the model's weights
3. Fine-tune on an instruction-following dataset with 🤗 `trl`'s `SFTTrainer`
4. Compare outputs before vs. after fine-tuning
5. Save the adapter, and optionally merge it back into the base model for deployment

### Key concepts

| Term | What it means |
|---|---|
| **Fine-tuning** | Continuing to train a pretrained model's weights on new, task-specific data. |
| **Full fine-tuning** | Updates *all* of a model's weights. Accurate but memory-hungry — a 7B model needs ~70GB+ of GPU RAM to train, far beyond a free Colab GPU. |
| **PEFT** | *Parameter-Efficient Fine-Tuning* — a family of methods that freeze the base model and train only a small number of extra parameters, cutting memory and compute needs by 10-100x. |
| **LoRA** | *Low-Rank Adaptation* — the most popular PEFT method. Instead of updating a weight matrix `W`, it learns two small matrices `A` and `B` such that `W + BA` approximates the fine-tuned weights. Only `A` and `B` are trained. |
| **QLoRA** | LoRA applied on top of a **4-bit quantized** base model. This is what lets us fine-tune a 1-7B parameter model on a free 16GB Colab GPU. |

### Before you start
Go to **Runtime > Change runtime type > Hardware accelerator** and select a **T4 GPU** (free tier). This notebook is sized to fit comfortably in T4's 16GB of VRAM.

> Colab sessions are ephemeral — if you want your fine-tuned adapter to survive a disconnect, mount Google Drive (see the optional cell below) or push it to the Hugging Face Hub.


## 1. Check your GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > Hardware accelerator > GPU (T4), then re-run this notebook.")


## 2. (Optional) Mount Google Drive

Useful so your saved adapter / checkpoints survive a Colab disconnect. Skip this cell if you don't need it.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = "/content/drive/MyDrive/peft-finetuning/lora-finetuned-model"  # then use this path below instead


## 3. Install dependencies

We deliberately do **not** pin exact old versions here — pinning to specific old releases can force `pip` to fall back to building packages like `tokenizers` or `numpy` from source (since no prebuilt wheel exists for Colab's current Python version), which fails with a `Failed building wheel` error. Installing the latest of each package avoids that, and this notebook's code targets the current `trl` (1.x) API.

If you hit a dependency error after running this cell, go to **Runtime > Restart session** and re-run from the top — a partially-installed environment is the most common cause of confusing import errors.


In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets huggingface_hub


## 4. Imports

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer


## 5. Configuration

`TinyLlama-1.1B-Chat` is used by default because it's small, fully open (no license gate), and trains quickly on a T4. Swap `MODEL_NAME` for something else if you like:

- `"Qwen/Qwen2.5-1.5B-Instruct"` — open, no gating
- `"meta-llama/Llama-3.2-1B-Instruct"` — requires accepting Meta's license on its Hugging Face model page, then logging in below
- `"google/gemma-2-2b-it"` — requires accepting Google's license on its Hugging Face model page, then logging in below


In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "timdettmers/openassistant-guanaco"  # classic small instruction-tuning dataset, already formatted as plain text
OUTPUT_DIR = "/content/lora-finetuned-model"
NEW_MODEL_NAME = "tinyllama-guanaco-lora"  # used if you push to the Hub

# --- LoRA hyperparameters ---
LORA_R = 16                 # rank of the LoRA update matrices — higher = more capacity, more params
LORA_ALPHA = 32             # scaling factor, usually ~2x LORA_R
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # attention projections (Llama-style architecture)

# --- Training hyperparameters ---
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = 250              # quick demo run; set to -1 to train for the full NUM_TRAIN_EPOCHS instead
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4          # effective batch size = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 512


## 6. (Optional) Hugging Face login

Only needed if you switched `MODEL_NAME` to a gated model (Llama, Gemma) above, or want to push your finished adapter to the Hub at the end.

In [ ]:
from huggingface_hub import notebook_login

# notebook_login()


## 7. Load and inspect the dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)
print("\n--- Example ---\n")
print(dataset["train"][0]["text"][:600])

# Use a subset for a fast demo run. Remove .select(...) (or increase the range)
# to train on more data once you've confirmed the pipeline works end to end.
train_dataset = dataset["train"].shuffle(seed=42).select(range(min(1000, len(dataset["train"]))))
print(f"\nTraining on {len(train_dataset)} examples")


## 8. Load the base model in 4-bit (QLoRA)

`BitsAndBytesConfig` quantizes the model's weights to 4 bits on load, which is what makes it possible to fit a billion-parameter model's weights, activations, optimizer state, and gradients all on a single free-tier T4 (16GB).

> T4 is a Turing-generation GPU and does not support `bfloat16` compute efficiently — use `torch.float16` here. If you're on an A100 (Colab Pro/Pro+), switch this to `torch.bfloat16` for better numerical stability.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.config.pretraining_tp = 1


## 9. Baseline: generate with the un-tuned model

We'll compare this output to the fine-tuned model's output later.

In [ ]:
def generate(model, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


test_prompt = "### Human: What is the difference between fine-tuning and PEFT? ### Assistant:"

baseline_output = generate(model, test_prompt)
print(baseline_output)


## 10. Attach LoRA adapters

`prepare_model_for_kbit_training` does some bookkeeping needed for training on top of a quantized model (casts norm layers to fp32, enables gradient checkpointing, etc.). `get_peft_model` then wraps the base model, freezes its original weights, and injects the small trainable LoRA matrices into the target modules.


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 11. Training arguments

`SFTConfig` (from `trl`) is a drop-in extension of 🤗 Transformers' `TrainingArguments` — it adds SFT-specific fields like `dataset_text_field`, `max_length`, and `packing` alongside the usual training hyperparameters, so everything lives in one config object.


In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",       # memory-efficient optimizer, pairs well with QLoRA
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)


## 12. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()


## 13. Save the adapter

Only the small LoRA adapter weights are saved here (a few MB to ~100MB), not the full base model.

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/final_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

# Optional: push the adapter to the Hugging Face Hub (requires notebook_login() above)
# trainer.model.push_to_hub(NEW_MODEL_NAME)
# tokenizer.push_to_hub(NEW_MODEL_NAME)


## 14. Compare: before vs. after fine-tuning

In [ ]:
finetuned_output = generate(model, test_prompt)

print("=" * 30, "BEFORE fine-tuning", "=" * 30)
print(baseline_output)
print("\n" + "=" * 30, "AFTER fine-tuning", "=" * 30)
print(finetuned_output)


## 15. (Optional) Merge the adapter into the base model

For deployment it's often more convenient to have one merged set of weights rather than a base model + separate adapter. This reloads the base model at full (non-quantized) `float16` precision and folds the LoRA weights into it.

> This step needs extra GPU memory since the base model is no longer 4-bit here. For very large base models, do this step on a machine with more VRAM, or skip it and load the base model + adapter together at inference time instead (`PeftModel.from_pretrained`).


In [ ]:
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

merged_model = PeftModel.from_pretrained(base_model_fp16, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

MERGED_DIR = f"{OUTPUT_DIR}/merged_model"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")


## Troubleshooting & tips

**Out of memory (OOM) errors**
- Lower `PER_DEVICE_BATCH_SIZE` (e.g. to 1 or 2) and raise `GRAD_ACCUM_STEPS` to compensate — this keeps the effective batch size the same while using less memory per step.
- Lower `MAX_SEQ_LENGTH` (e.g. to 256).
- Make sure `gradient_checkpointing=True` and you're using the 4-bit `bnb_config` (don't skip step 8).
- Try a smaller model, or reduce `LORA_R`.

**"Gated repo" / 403 error when loading a model**
- Some models (Llama, Gemma) require accepting a license on their Hugging Face model page while logged in, then authenticating in this notebook with `notebook_login()` (step 6) using a token that has been granted access.

**`TypeError: unexpected keyword argument` from `SFTConfig`/`SFTTrainer`**
- `trl` occasionally renames fields between versions (this happened to `max_seq_length` → `max_length`, and `tokenizer=` → `processing_class=`, both already reflected in this notebook). If you installed a very new or very old `trl` and hit this, run `from trl import SFTConfig; help(SFTConfig)` in a cell to see the current field names on your installed version, and adjust step 11/12 accordingly.

**Training seems too slow**
- Lower `MAX_STEPS`, or shrink the training subset in step 7 (`.select(range(...))`).
- A free Colab T4 is shared and can vary in speed; Colab Pro gives priority access to faster GPUs (A100/L4).

**Session disconnected and I lost my progress**
- Colab free-tier sessions can disconnect after a period of inactivity or a fixed time limit. Mount Google Drive (step 2) and point `OUTPUT_DIR` there, or push adapters to the Hugging Face Hub periodically, so your work persists across sessions.

**Want to fine-tune on your own data instead?**
- Replace `DATASET_NAME` with a dataset id from the Hugging Face Hub, or build a local `datasets.Dataset` from a list of `{"text": "..."}` dicts using `Dataset.from_list(...)`. Keep each example formatted as a single `text` field containing your prompt + desired completion, matching whatever chat/instruction template your model expects.

### References
- LoRA paper: https://arxiv.org/abs/2106.09685
- QLoRA paper: https://arxiv.org/abs/2305.14314
- Hugging Face PEFT docs: https://huggingface.co/docs/peft
- Hugging Face TRL (SFTTrainer) docs: https://huggingface.co/docs/trl
